In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Read all the words

words = open('names.txt', 'r').read().splitlines()
words[:8]

['frazier',
 'miller',
 'gilbert',
 'ward',
 'arnold',
 'saunders',
 'mcdaniel',
 'collins']

In [3]:
len(words)

46654

In [4]:
# Build the vocalbulory of characters and mappings to/from integers

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos) 

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


Build the dataset


In [127]:
# Build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], [] # X are the input and Y are the labels for each example inside X
for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '------->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)


In [128]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([329934, 3]), torch.int64, torch.Size([329934]), torch.int64)

----

In [62]:
C = torch.randn(27, 2)

In [63]:
emd = C[X]
emd.shape

torch.Size([35, 3, 2])

In [97]:
W1 = torch.randn(6, 100)
b1 = torch.randn(100)

In [65]:
# torch.cat([emd[:, 0, :], emd[:, 1, :], emd[:, 2, :]], 1).shape 

# this can't be change with changing block_size

In [66]:
# torch.cat(torch.unbind(emd,1), 1).shape # another way to do it

In [67]:
h = torch.tanh(emd.view(-1, 6) @ W1 + b1)

In [68]:
h.shape

torch.Size([35, 100])

In [69]:
W2 = torch.randn(100, 27)
b2 = torch.randn(27)

In [70]:
logits = h @ W2 + b2

In [71]:
logits.shape

torch.Size([35, 27])

In [73]:
counts = logits.exp()

In [74]:
probs = counts / counts.sum(1, keepdim=True)

In [75]:
probs.shape

torch.Size([35, 27])

In [77]:
probs[0].sum()

tensor(1.0000)

In [85]:
loss = -probs[torch.arange(35), Y].log().mean()
loss

tensor(18.4386)

---

*organised code*

In [129]:
X.shape, Y.shape #dataset

(torch.Size([329934, 3]), torch.Size([329934]))

In [130]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn((100), generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [131]:
sum(p.nelement() for p in parameters) # numbers of parameter intotal)

3481

In [132]:
"""
# forward pass

emd = C[X] # (32, 3, 2)
h = torch.tanh(emd.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)

# counts = logits.exp()
# probs = counts / counts.sum(1, keepdim=True)
#loss = -probs[torch.arange(35), Y].log().mean()

# ⬆⬆⬆ this can be replaced by ⬇⬇⬇ ---> F.cross_entropy(logits, Y)

loss = F.cross_entropy(logits, Y)
loss

"""

'\n# forward pass\n\nemd = C[X] # (32, 3, 2)\nh = torch.tanh(emd.view(-1, 6) @ W1 + b1) # (32, 100)\nlogits = h @ W2 + b2 # (32, 27)\n\n# counts = logits.exp()\n# probs = counts / counts.sum(1, keepdim=True)\n#loss = -probs[torch.arange(35), Y].log().mean()\n\n# ⬆⬆⬆ this can be replaced by ⬇⬇⬇ ---> F.cross_entropy(logits, Y)\n\nloss = F.cross_entropy(logits, Y)\nloss\n\n'

### CrossEntropyLoss in PyTorch

#### torch.nn.functional.cross_entropy(input, target, weight=None, size_average=None, ignore_index=-100, reduce=None, reduction='mean', label_smoothing=0.0)[source]
#### Compute the cross entropy loss between input logits and target.


*Advantages*
- PyTorch cluster up all the operations and makes fused kernal that very effiently evalue this operations.
- Backaward can be much more efficent (not just cuz of its fused kernal), but analytically and mathematically its often much simpler backward pass.
- Under the hood, F.cross_entropy can also be significantly more numerically well-behaved; we cannot pass very large logits throught the manual expression.



In [133]:
for p in parameters:
    p.requires_grad = True

In [ ]:
for _ in range(10):

    # mini-batch construct
    ix = torch.randint(0, X.shape[0], (32,))

    #forward pass
    emd = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emd.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad 

print(loss.item())



2.9574761390686035


In [163]:
emd = C[X] # (32, 3, 2)
h = torch.tanh(emd.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y)
loss

tensor(2.8196, grad_fn=<NllLossBackward0>)